In [ ]:
import face_recognition 
import cv2 as cv

ModuleNotFoundError: No module named 'face_recognition'

In [ ]:
img1 = r"C:\Users\ANASGROUP\Pictures\WhatsApp_Image_2025-05-11_at_23_LE_upscale_balanced_x4.jpg"
img2 = r"C:\Users\ANASGROUP\Pictures\WhatsApp Image 2025-06-11 at 04.00.04_5afadfd1.jpg"
img1	= cv.imread(img1)	
img2	= cv.imread(img2)	
img1 = cv.cvtColor(img1, cv.COLOR_BGR2RGB)
img2 = cv.cvtColor(img2, cv.COLOR_BGR2RGB)

face_loc2 = face_recognition.face_locations(img2)[0]
encode_face2 = face_recognition.face_encodings(img2)[0]
face_loc = face_recognition.face_locations(img1)[0]
encode_face1 = face_recognition.face_encodings(img1)[0]

result = face_recognition.compare_faces([encode_face1], encode_face2)
print(result)

[True]


In [3]:
pwd

'd:\\deeplearning_course\\read_image_write\\face_recognation'

In [2]:
import cv2 as cv 
import numpy as np	
import glob 
import os 
import face_recognition
import onnxruntime as ort
from utils import calculate_distance, play_alarm, stop_alarm	 
from utils import predict_tensor 
from utils import preprocess_image 
import time 
import torch
import pygame
import torch.nn as nn
from torchvision import models
from torchvision import transforms


ModuleNotFoundError: No module named 'face_recognition'

In [2]:
class SimpleFace : 
    def __init__(self):
        self.face_encodeing = []
        self.face_names = []
        self.frame_resizing = 0.25				

    def load_encoding_images(self, images_path):
        images_path = glob.glob(os.path.join(images_path, "*.*"))
        for img_path in images_path:
            img = cv.imread(img_path)	
            rgb_img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
            encodings = face_recognition.face_encodings(rgb_img)
            if len(encodings) == 0:
                continue
            encode = encodings[0]
            self.face_encodeing.append(encode)
            basename = os.path.basename(img_path)	# this will give us the name of the image file	and extension	
            #so split the name about the extension and take the first part as the name of the person in the image	
            (self.face_names).append(os.path.splitext(basename)[0])	# (name , extension	)

    def detect_known_faces(self, frame):
        small_frame = cv.resize(
            frame,
            (0, 0),
            fx=self.frame_resizing,
            fy=self.frame_resizing
        )  # to speed up the processing of the frame we will resize it to 1/4 of its original size	
        rgb_small_frame = cv.cvtColor(small_frame, cv.COLOR_BGR2RGB)
        face_locations = face_recognition.face_locations(rgb_small_frame)  # to know the location of face 
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)
        face_names = []
        for face_encoding in face_encodings:
            matches = face_recognition.compare_faces(self.face_encodeing, face_encoding)
            name = "Unknown"
            if True in matches:
                match_index = matches.index(True)
                name = self.face_names[match_index]
            face_names.append(name)
        return face_locations, face_names

In [6]:
pwd

'd:\\deeplearning_course\\read_image_write\\face_recognation'

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5])
])

In [4]:
prototext = r"D:\deeplearning_course\read_image_write\real_time_mask_detection\deploy.prototxt.txt"
weights_path = r"D:\deeplearning_course\read_image_write\real_time_mask_detection\res10_300x300_ssd_iter_140000.caffemodel"
faceNet = cv.dnn.readNet(prototext , weights_path)

In [8]:
import cv2 as cv
import numpy as np
from utils import generate_event
from utils import send_event 
from utils import detect_face
face_recognizer = SimpleFace()
database_encoded = face_recognizer.load_encoding_images(
    r"D:\deeplearning_course\read_image_write\face_images"
)
distance_list =[]
cap = cv.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    _, face_names = face_recognizer.detect_known_faces(frame)
    face_locations = detect_face(frame, faceNet)	
    for face_loc, name in zip(face_locations, face_names):
        x1 , y1 ,x2 ,y2 = [v for v in face_loc]
        distance_cm = calculate_distance((y1, x2, y2, x1))
        distance_list.append(distance_cm)	
        
       
        face = frame[y1:y2, x1:x2]
        if face.size == 0:
            continue

       
        face_input  = preprocess_image(face)
        label, real_prob, spoof_prob = predict_tensor(face_input)
        
        if spoof_prob > 0.5:
            text = f"Spoof: {spoof_prob:.2f}"
            color = (0, 0, 255)
            play_alarm()	
        else:
            text = f"Live: {real_prob:.2f}"
            color = (0, 255, 0) 
            
            stop_alarm()
            if len(distance_list) > 1 and distance_list[len(distance_list)-1]-distance_list [len(distance_list)-2] > 10:	
                 play_alarm()
            event = generate_event(name, distance_cm, real_prob)	
            send_event(event)
       
       
        cv.putText(frame, text, (x1, y1 - 30), cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        cv.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv.putText(frame, name, (x1, y1 - 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    cv.imshow("Frame", frame)
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv.destroyAllWindows()

[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for Unknown
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Event sent for ahmedkoka
[INFO] Eve

In [11]:
import cv2
import os
import time
import face_recognition  # لاكتشاف الوجه

# إعداد مجلد حفظ الصور
base_dir = r"D:\deeplearning_course\read_image_write\face_dataset"
real_dir = os.path.join(base_dir, "real")
os.makedirs(real_dir, exist_ok=True)

cap = cv2.VideoCapture(0)

real_count = 300
max_real = 500 
save_interval = 1   
last_save_time = time.time()

print("ابدأ بوضع صور Real في الكاميرا، سيتم حفظ أول 150 صورة كل 1 ثواني.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # اكتشاف الوجه
    face_locations = face_recognition.face_locations(frame)

    for face_loc in face_locations:
        y1, x2, y2, x1 = face_loc
        HEAD_UP = 30    # زيادة أعلى الرأس
        HEAD_DOWN = 20  # زيادة أسفل الرأس
        
        h, w, _ = frame.shape
        
        # ضبط الاحداثيات
        y1_up   = max(0, y1 - HEAD_UP)
        y2_down = min(h, y2 + HEAD_DOWN)
        x1      = max(0, x1)
        x2      = min(w, x2)
        
        # Crop كامل للرأس
        face = frame[y1_up:y2_down, x1:x2]

        if face.size == 0:
            continue

        # حفظ الصورة كل save_interval ثانية
        if time.time() - last_save_time > save_interval:
            last_save_time = time.time()

            if real_count < max_real:
                filename = os.path.join(real_dir, f"real_{real_count:03d}.jpg")
                cv2.imwrite(filename, face)
                real_count += 1
                print(f"Saved {filename}")

            if real_count >= max_real:
                print("اكتملت جمع 150 صورة real.")
                cap.release()
                cv2.destroyAllWindows()
                exit()

    # عرض الكاميرا
    cv2.imshow("Frame", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

ابدأ بوضع صور Real في الكاميرا، سيتم حفظ أول 150 صورة كل 1 ثواني.
Saved D:\deeplearning_course\read_image_write\face_dataset\real\real_300.jpg
Saved D:\deeplearning_course\read_image_write\face_dataset\real\real_301.jpg
Saved D:\deeplearning_course\read_image_write\face_dataset\real\real_302.jpg
Saved D:\deeplearning_course\read_image_write\face_dataset\real\real_303.jpg
Saved D:\deeplearning_course\read_image_write\face_dataset\real\real_304.jpg
Saved D:\deeplearning_course\read_image_write\face_dataset\real\real_305.jpg
Saved D:\deeplearning_course\read_image_write\face_dataset\real\real_306.jpg
Saved D:\deeplearning_course\read_image_write\face_dataset\real\real_307.jpg
Saved D:\deeplearning_course\read_image_write\face_dataset\real\real_308.jpg
Saved D:\deeplearning_course\read_image_write\face_dataset\real\real_309.jpg
Saved D:\deeplearning_course\read_image_write\face_dataset\real\real_310.jpg
Saved D:\deeplearning_course\read_image_write\face_dataset\real\real_311.jpg
Saved D:\d

In [10]:
face_names

['ahmedkoka']